# TESTE DE ESTRESSE DE HARDWARE: PANDAS (~2 GB RAM) VS. POLARS


## Objetivo do Experimento (Aquecimento para a Aula 12)

Antes de manipularmos bases reais de gigabytes na **Aula 12**, vamos fazer um experimento prático ao vivo para enxergar no **Gerenciador de Tarefas do Windows** o que acontece dentro do computador quando criamos uma tabela gigante de **14 milhões de registros**:
1. Quanto de **Memória RAM** o **Pandas** consome (vai bater perto de **2,15 GB**!) versus quanto o **Polars** consome para guardar exatamente os mesmos dados.
2. Como os **Núcleos do Processador (CPU)** trabalham em cada biblioteca (1 núcleo no Pandas vs. todos os núcleos no Polars).
3. Como limpar a memória RAM corretamente (`del` + `gc.collect()`) para não travar a estação de trabalho.

## Passo a Passo para Acompanhar no Gerenciador de Tarefas do Windows

1. **Feche programas pesados** antes de começar (navegadores com muitas abas, Teams, etc.), conforme orienta o nosso Plano de Curso.
2. Pressione **`Ctrl + Shift + Esc`** no teclado para abrir o **Gerenciador de Tarefas**.
3. Clique na aba **Desempenho** (ícone de gráfico no menu esquerdo) e selecione **Memória** para ver o consumo total em GB em tempo real. Deixe a janela visível ao lado do VS Code.
   - *Dica extra:* Clique também em **CPU**, clique com o botão direito sobre o gráfico principal e escolha **Alterar gráfico para $\rightarrow$ Processadores lógicos** para enxergar cada núcleo trabalhando!
4. **Execute a Célula 1 (Importações)** e anote quanto de RAM o Windows está usando antes do teste.
5. **Execute a Célula 2 (Teste com Pandas)** e observe o degrau de **+2,15 GB** subindo na barra de Memória RAM.
6. **Execute a Célula 3 (Limpeza da RAM)** e veja a memória voltar ao normal.
7. **Execute a Célula 4 (Teste com Polars)** e compare o degrau muito menor na RAM e o pico simultâneo nos núcleos da CPU!

In [1]:
# ==============================================================================
# CÉLULA 1: PREPARAÇÃO DO AMBIENTE E FUNÇÃO DE MONITORAMENTO DE RAM
# ==============================================================================

# Importa NumPy para gerar os vetores numéricos de base
import numpy as np

# Importa Pandas e Polars para o comparativo de memória e velocidade
import pandas as pd
import polars as pl

# Importa 'time' para cronometrar os segundos e 'gc' (Garbage Collector) para limpar a RAM
import time
import gc

# Importa 'os' e 'psutil' para ler exatamente quanto de RAM o processo Python está usando
import os
import psutil

# Cria uma função auxiliar que consulta o sistema operacional e retorna a RAM usada em GB
def ram_atual_gb():
    processo = psutil.Process(os.getpid())              # Pega o ID do processo Python atual
    bytes_usados = processo.memory_info().rss           # Lê a memória física alocada (RSS)
    return bytes_usados / (1024 ** 3)                   # Converte de Bytes para Gigabytes (GB)

# Define o tamanho da base: 14 milhões de linhas (~2,15 GB no Pandas)
N_LINHAS = 14_000_000

print(f"Ambiente pronto! RAM inicial ocupada pelo Python: {ram_atual_gb():.2f} GB")
print("Abra o Gerenciador de Tarefas (Ctrl + Shift + Esc) na aba Desempenho -> Memória!")

Ambiente pronto! RAM inicial ocupada pelo Python: 0.11 GB
Abra o Gerenciador de Tarefas (Ctrl + Shift + Esc) na aba Desempenho -> Memória!


In [2]:
# ==============================================================================
# CÉLULA 2: CRIANDO A BASE DE ~2,15 GB NO PANDAS (OLHE O GERENCIADOR AGORA!)
# ==============================================================================

# Fixa a semente aleatória para reprodutibilidade
np.random.seed(42)

# Marca o tempo inicial e a RAM antes de criar a tabela
ram_antes_pd = ram_atual_gb()
t0_pd = time.time()

# Cria o DataFrame Pandas com 14 milhões de linhas (5 colunas numéricas + 2 colunas de texto)
df_pandas_gigante = pd.DataFrame({
    'ID_Beneficiario': np.arange(N_LINHAS, dtype=np.int64),
    'Valor_Parcela': np.random.uniform(150.0, 1200.0, size=N_LINHAS),
    'Renda_Per_Capita': np.random.uniform(0.0, 450.0, size=N_LINHAS),
    'Qtd_Dependentes': np.random.randint(0, 8, size=N_LINHAS, dtype=np.int64),
    'Mes_Competencia': np.random.randint(1, 13, size=N_LINHAS, dtype=np.int64),
    'UF': np.random.choice(['RJ', 'SP', 'MG', 'BA', 'PE', 'CE', 'PR', 'RS'], size=N_LINHAS),
    'Municipio': np.random.choice(['Rio de Janeiro', 'Sao Paulo', 'Belo Horizonte', 'Salvador', 'Fortaleza', 'Recife'], size=N_LINHAS)
})

# Calcula o tempo de criação e o tamanho exato do DataFrame na memória RAM
tempo_criacao_pd = time.time() - t0_pd
tamanho_df_pd_gb = df_pandas_gigante.memory_usage(deep=True).sum() / (1024 ** 3)
ram_depois_pd = ram_atual_gb()

print(f"[PANDAS] Tempo para criar 14 milhões de linhas: {tempo_criacao_pd:.2f} segundos")
print(f"[PANDAS] Tamanho exato da tabela na RAM:        {tamanho_df_pd_gb:.2f} GB")
print(f"[PANDAS] RAM total do processo Python agora:    {ram_depois_pd:.2f} GB (Subiu +{ram_depois_pd - ram_antes_pd:.2f} GB)")
print("\n--> Observe o degrau no Gerenciador de Tarefas e depois rode a Célula 3 para limpar!")

[PANDAS] Tempo para criar 14 milhões de linhas: 2.85 segundos
[PANDAS] Tamanho exato da tabela na RAM:        1.96 GB
[PANDAS] RAM total do processo Python agora:    2.27 GB (Subiu +2.16 GB)

--> Observe o degrau no Gerenciador de Tarefas e depois rode a Célula 3 para limpar!


In [3]:
# ==============================================================================
# CÉLULA 3: LIMPANDO A MEMÓRIA RAM ANTES DO TESTE COM POLARS
# ==============================================================================

# Apaga a variável gigante da memória
del df_pandas_gigante

# Força o coletor de lixo (Garbage Collector) a devolver a memória RAM para o Windows
gc.collect()

print(f"Memória limpa! RAM atual do processo Python: {ram_atual_gb():.2f} GB")
print("Observe no Gerenciador de Tarefas que o gráfico de Memória desceu!")

Memória limpa! RAM atual do processo Python: 0.11 GB
Observe no Gerenciador de Tarefas que o gráfico de Memória desceu!


In [4]:
# ==============================================================================
# CÉLULA 4: CRIANDO EXATAMENTE A MESMA BASE NO POLARS (COMPARE NO GERENCIADOR!)
# ==============================================================================

# Fixa a mesma semente aleatória para gerar exatamente os mesmos 14 milhões de registros
np.random.seed(42)

# Marca o tempo inicial e a RAM antes de criar a tabela no Polars
ram_antes_pl = ram_atual_gb()
t0_pl = time.time()

# Cria o DataFrame Polars (memória colunar Apache Arrow, sem índice pesado)
df_polars_gigante = pl.DataFrame({
    'ID_Beneficiario': np.arange(N_LINHAS, dtype=np.int64),
    'Valor_Parcela': np.random.uniform(150.0, 1200.0, size=N_LINHAS),
    'Renda_Per_Capita': np.random.uniform(0.0, 450.0, size=N_LINHAS),
    'Qtd_Dependentes': np.random.randint(0, 8, size=N_LINHAS, dtype=np.int64),
    'Mes_Competencia': np.random.randint(1, 13, size=N_LINHAS, dtype=np.int64),
    'UF': np.random.choice(['RJ', 'SP', 'MG', 'BA', 'PE', 'CE', 'PR', 'RS'], size=N_LINHAS),
    'Municipio': np.random.choice(['Rio de Janeiro', 'Sao Paulo', 'Belo Horizonte', 'Salvador', 'Fortaleza', 'Recife'], size=N_LINHAS)
})

# Calcula o tempo de criação e o tamanho exato da tabela Polars na memória RAM
tempo_criacao_pl = time.time() - t0_pl
tamanho_df_pl_gb = df_polars_gigante.estimated_size() / (1024 ** 3)
ram_depois_pl = ram_atual_gb()

print(f"[POLARS] Tempo para criar 14 milhões de linhas: {tempo_criacao_pl:.2f} segundos")
print(f"[POLARS] Tamanho exato da tabela na RAM:        {tamanho_df_pl_gb:.2f} GB")
print(f"[POLARS] RAM total do processo Python agora:    {ram_depois_pl:.2f} GB (Subiu +{ram_depois_pl - ram_antes_pl:.2f} GB)")
print("-" * 65)
print(f"RESUMO: O Polars ocupou {tamanho_df_pd_gb / tamanho_df_pl_gb:.1f}x MENOS memória RAM que o Pandas!")

[POLARS] Tempo para criar 14 milhões de linhas: 4.82 segundos
[POLARS] Tamanho exato da tabela na RAM:        0.68 GB
[POLARS] RAM total do processo Python agora:    1.11 GB (Subiu +1.00 GB)
-----------------------------------------------------------------
RESUMO: O Polars ocupou 2.9x MENOS memória RAM que o Pandas!


In [5]:
# ==============================================================================
# CÉLULA 5: LIMPEZA FINAL DA MEMÓRIA APÓS O EXPERIMENTO
# ==============================================================================
del df_polars_gigante
gc.collect()
print(f"Experimento concluído! RAM liberada: {ram_atual_gb():.2f} GB")

Experimento concluído! RAM liberada: 0.59 GB


*Leitura Analítica do Experimento:*

1. **Por que o Pandas ocupou ~2,15 GB e o Polars ocupou apenas ~0,75 GB?** Porque o Pandas armazena cada texto repetido (`'Rio de Janeiro'`, `'Sudeste'`) como um objeto Python separado na memória e ainda cria um índice numérico pesado. Já o Polars usa o formato colunar **Apache Arrow**, compactando textos e vetores numéricos diretamente na memória nativa.    
   
2. **Por que a Célula 3 (`del` + `gc.collect()`) é obrigatória em Big Data?** Em projetos de larga escala, variáveis antigas ficam presas na memória RAM. Apagar tabelas intermediárias que não serão mais usadas evita que o computador trave por falta de memória.